### Imports

In [1]:
import numpy as np
import pandas as pd
import joblib
import json

from scipy.optimize import differential_evolution

### Load metadata

In [2]:
with open("artifacts/feature_cols.json", "r") as f:
    feature_cols = json.load(f)

print("metadata loaded")

metadata loaded


### Train the model again

In [3]:
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, median_absolute_error,
    r2_score, explained_variance_score, max_error
)
from scipy.stats import pearsonr

def eval_reg(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    mxerr = max_error(y_true, y_pred)

    # --- percentage errors (safe for zeros) ---
    denom = np.where(y_true == 0, np.nan, np.abs(y_true))
    mape = np.nanmean(np.abs((y_true - y_pred) / denom)) * 100

    smape_denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    smape = np.nanmean(
        np.where(smape_denom == 0, np.nan,
                 np.abs(y_true - y_pred) / smape_denom)
    ) * 100

    # --- correlation ---
    try:
        pearson_r = pearsonr(y_true, y_pred)[0]
    except Exception:
        pearson_r = np.nan

    return {
        "rmse": rmse,
        "mae": mae,
        "medae": medae,
        "r2": r2,
        "explained_variance": evs,
        "max_error": mxerr,
        "mape_%": mape,
        "smape_%": smape,
        "pearson_r": pearson_r,
    }

### Load and prepare data

In [4]:
TRAIN_PATH = '.../modeling/meal_dataset_train_subject_split.csv'
TEST_PATH  = '.../modeling/meal_dataset_test_subject_split.csv'
SUPP_PATH = '.../modeling/meal_modeling_dataset_all_subjects_E4_fixed_mealtimes.csv'

def load_data():
    train = pd.read_csv(TRAIN_PATH)
    test  = pd.read_csv(TEST_PATH)
    supp  = pd.read_csv(SUPP_PATH)

    DROP_COLS = [
        "subject", "meal_ts", "cluster_start_ts",
        "cluster_end_ts", "cluster_n_events"
    ]
    DROP_FEATURE_COLS = ["met_mean_2h", "met_std_2h", "sex"]
    TARGET = "y_max_bgl_2h_post_mgdl"

    train = train.drop(columns=DROP_COLS, errors="ignore")
    test  = test.drop(columns=DROP_COLS, errors="ignore")

    train = train.drop(columns=DROP_FEATURE_COLS, errors="ignore")
    test  = test.drop(columns=DROP_FEATURE_COLS, errors="ignore")
    supp  = supp.drop(columns=DROP_FEATURE_COLS, errors="ignore")

    supp = supp[train.columns]
    train = pd.concat([train, supp], axis=0)

    train = train.replace([np.inf, -np.inf], np.nan).dropna()
    test  = test.replace([np.inf, -np.inf], np.nan).dropna()
    
    X_train_full = train.drop(columns=[TARGET])
    y_train_full = train[TARGET].values

    X_test = test.drop(columns=[TARGET])
    y_test = test[TARGET].values

    # enforce same column order
    X_test = X_test[feature_cols]

    return X_train_full, y_train_full, X_test, y_test


X_train_full, y_train_full, X_test, y_test = load_data()

In [5]:
from lightgbm import LGBMRegressor

# --------------------------------------------------
# Best params (fixed)
# --------------------------------------------------
lgb_best_params = {
    "n_estimators": 439,
    "learning_rate": 0.0875390351800604,
    "num_leaves": 43,
    "max_depth": 4,
    "min_child_samples": 28,
    "subsample": 0.848553073033381,
    "colsample_bytree": 0.7039794883479599,
    "reg_alpha": 0.020914981329035603,
    "reg_lambda": 0.002323350351539011,
    "min_split_gain": 0.0001116602913398135,
}

mono = [0] * len(feature_cols)

def set_mono(col, val):
    if col in feature_cols:
        mono[feature_cols.index(col)] = val

set_mono('carbs_g_cluster_sum', +1)
set_mono('pre_meal_bgl_mgdl', +1)

# --------------------------------------------------
# Train final model
# --------------------------------------------------
model = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=-1,
    monotone_constraints=mono,
    force_col_wise=True,
    verbosity=-1,
    **lgb_best_params,
)

print("Training LightGBM with fixed params...")
model.fit(X_train_full, y_train_full)

# --------------------------------------------------
# Evaluate
# --------------------------------------------------
pred_test = model.predict(X_test)

metrics = eval_reg(y_test, pred_test)

print("\n📊 LightGBM Test Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, (float, np.floating)) else f"{k}: {v}")

Training LightGBM with fixed params...

📊 LightGBM Test Metrics:
rmse: 17.5586
mae: 13.6515
medae: 11.5006
r2: 0.3988
explained_variance: 0.4019
max_error: 63.2524
mape_%: 9.6297
smape_%: 9.6634
pearson_r: 0.6434


### Counterfactual functions

In [6]:
EPS = 1e-6

C_COL = 'carbs_g_cluster_sum'
P_COL = 'protein_g_cluster_sum'
F_COL = 'fat_g_cluster_sum'


def predict_one(model, x):
    if isinstance(x, pd.Series):
        x = x.to_frame().T
    return float(model.predict(x)[0])


def generate_cf_for_row(
    model,
    x_row,
    target_threshold=145.0,
    target_cap=140.0,
    lambda_reg=1,
    p_max=200.0,
    f_max=200.0,
    maxiter=80,
    popsize=15,
):
    y0 = predict_one(model, x_row)

    if y0 < target_threshold:
        return {"status": "skip", "y_pred_orig": y0}

    C0, P0, F0 = x_row[C_COL], x_row[P_COL], x_row[F_COL]
    m0 = np.array([C0, P0, F0])

    scale = np.maximum(m0, 10.0)

    def objective(m):
        C, P, F = m
        xr = x_row.copy()
        xr[C_COL], xr[P_COL], xr[F_COL] = C, P, F

        y = predict_one(model, xr)

        loss_target = max(0, y - target_cap) ** 2
        loss_close = lambda_reg * np.sum(((m - m0) / scale) ** 2)

        return loss_target + loss_close

    bounds = [
        (EPS, max(EPS, C0)),
        (EPS, p_max),
        (EPS, f_max),
    ]

    res = differential_evolution(
        objective,
        bounds,
        maxiter=maxiter,
        popsize=popsize,
        seed=42,
        polish=True,
    )

    m_cf = res.x

    xr_cf = x_row.copy()
    xr_cf[C_COL], xr_cf[P_COL], xr_cf[F_COL] = m_cf

    y_cf = predict_one(model, xr_cf)

    return {
        "status": "ok",
        "y_pred_orig": float(y0),
        "y_pred_cf": float(y_cf),
        "carbs_orig": C0,
        "carbs_cf": m_cf[0],
        "protein_orig": P0,
        "protein_cf": m_cf[1],
        "fat_orig": F0,
        "fat_cf": m_cf[2],
    }

In [7]:
results = []

for i in range(len(X_test)):
    out = generate_cf_for_row(model, X_test.iloc[i])
    out["row_idx"] = i
    out["y_true"] = float(y_test[i])
    results.append(out)

cf_df = pd.DataFrame(results)

cf_df.to_csv("artifacts/counterfactuals.csv", index=False)
print("Saved counterfactuals")

Saved counterfactuals


### Summary

In [8]:
ok = cf_df[cf_df.status == "ok"].copy()

if len(ok):
    ok["delta_pred"] = ok["y_pred_cf"] - ok["y_pred_orig"]
    ok["delta_carbs"] = ok["carbs_cf"] - ok["carbs_orig"]

    print("CF generated:", len(ok))
    print("Mean drop:", (-ok["delta_pred"]).mean())
    #print("Success rate (<=140):", (ok["y_pred_cf"] <= 143).mean())

CF generated: 50
Mean drop: 12.78647777601501
